In [2]:
import tkinter as tk
from tkinter import ttk
import time
import random
BANGS = ["WA", "NT", "SA", "Q", "NSW", "V", "T"]
MAU_SAC = ["Red", "Green", "Blue"]
MAU_HEX = {"Red": "#ff4d4d", "Green": "#2ecc71", "Blue": "#3498db", None: "#f2f2f2"}

class GiaoDienToMauAndOrGiatCap:
    def __init__(self, cua_so):
        self.cua_so = cua_so
        self.cua_so.title("Map Coloring - Forward Checking Search")

        self.phan_bo_mau = {bang: None for bang in BANGS}
        self.so_buoc = 0
        self.dang_chay = False
        self.dang_tam_dung = False
        self.bi_ket = False 
        
        self.cac_vung_giao_dien = {}
        self.toa_do_hien_tai = {}
        self.giap_ranh_dong = {} 
        khung_tren = tk.Frame(cua_so)
        khung_tren.pack(pady=10, fill="x")

        self.nhan_trang_thai = tk.Label(
            khung_tren, text="Trạng thái: Sẵn sàng. Nhấn 'Đặt Lại' để sinh địa hình và tính lại ranh giới!", anchor="w", font=("Arial", 11)
        )
        self.nhan_trang_thai.pack(side="left", padx=10, fill="x", expand=True)

        self.btn_chay = tk.Button(khung_tren, text="Chạy Forward Checking", command=self.bat_dau_thuat_toan, bg="#007acc", fg="white", font=("Arial", 10, "bold"))
        self.btn_chay.pack(side="right", padx=5)
        
        self.btn_dung = tk.Button(khung_tren, text="Dừng thuật toán", command=self.dao_trang_thai_dung, state="disabled", bg="#e74c3c", fg="white")
        self.btn_dung.pack(side="right", padx=5)
        
        tk.Button(khung_tren, text="Đặt Lại", command=self.dat_lai_va_sinh_dia_hinh).pack(side="right", padx=5)
        khung_chinh = tk.Frame(cua_so)
        khung_chinh.pack(padx=10, pady=10, fill="both", expand=True)

        khung_trai = tk.Frame(khung_chinh)
        khung_trai.pack(side="left", anchor="n")

        self.canvas = tk.Canvas(khung_trai, width=440, height=480, bg="#eef2f5", relief="raised", borderwidth=2)
        self.canvas.pack()

        khung_quy_tac = tk.Frame(khung_trai, bg="#34495e", padx=10, pady=10)
        khung_quy_tac.pack(pady=15, fill="x")
        tk.Label(khung_quy_tac, text="DYNAMIC GEOMETRY DETECTION", font=("Arial", 10, "bold"), bg="#34495e", fg="white").pack()
        self.txt_rule = tk.Label(khung_quy_tac, text="Địa hình co giãn & đổi vị trí hỗn loạn.\nCode tự quét tọa độ để cập nhật luật giáp ranh!", font=("Arial", 9), bg="#34495e", fg="#bdc3c7")
        self.txt_rule.pack()

        khung_phai = tk.Frame(khung_chinh, padx=10)
        khung_phai.pack(side="left", fill="both", expand=True)
        tk.Label(khung_phai, text="LỘ TRÌNH NHÌN TRƯỚC TRỰC QUAN (FORWARD CHECKING LOG)", font=("Arial", 10, "bold"), fg="#007acc").pack(anchor="w", pady=2)
        
        thanh_cuon = tk.Scrollbar(khung_phai)
        thanh_cuon.pack(side="right", fill="y")

        self.o_chu_log = tk.Text(
            khung_phai, width=45, height=25, font=("Courier", 10, "bold"),
            bg="#fdfdfd", yscrollcommand=thanh_cuon.set
        )
        self.o_chu_log.pack(side="left", fill="both", expand=True)
        thanh_cuon.config(command=self.o_chu_log.yview)
        
        self.dat_lai_va_sinh_dia_hinh()

    def sinh_ban_do_so_le_tu_nhien(self):
        w_trai = random.randint(110, 140); w_giua = random.randint(110, 135); w_phai = random.randint(110, 135)
        h_hang1 = random.randint(100, 125); h_hang2 = random.randint(110, 135); h_hang3 = random.randint(75, 95)
        dy_trai = random.randint(10, 35); dy_giua = random.randint(-15, 15); dy_phai = random.randint(15, 40)
        dx_sa = random.randint(-20, 20); dx_nsw = random.randint(10, 30); dx_v = random.randint(-25, 25)
        x_goc, y_goc = 40, 50
        x1_l, y1_l = x_goc, y_goc + dy_trai
        x2_l, y2_l = x1_l + w_trai, y1_l + h_hang1 + h_hang2
        x1_m1, y1_m1 = x2_l, y_goc + dy_giua
        x2_m1, y2_m1 = x1_m1 + w_giua, y1_m1 + h_hang1
        x1_r1, y1_r1 = x2_m1, y_goc + dy_phai
        x2_r1, y2_r1 = x1_r1 + w_phai, y1_r1 + h_hang1 + 20
        x1_m2, y1_m2 = x2_l + dx_sa, y2_m1
        x2_m2, y2_m2 = x2_m1 + 10, y1_m2 + h_hang2
        x1_r2, y1_r2 = x2_m2, y2_r1
        x2_r2, y2_r2 = x2_r1 - dx_nsw, y1_r2 + (y2_m2 - y1_r2) + 15
        x1_b, y1_b = x2_l + dx_v, y2_m2
        x2_b, y2_b = x2_r2, y1_b + h_hang3
        x1_t, y1_t = x1_b + random.randint(30, 70), y2_b + 15
        x2_t, y2_t = x1_t + random.randint(110, 160), y1_t + 45
        vung_dat_phat_sinh = [
            [x1_l, y1_l, x2_l, y1_l, x2_l, y2_l, x1_l, y2_l],
            [x1_m1, y1_m1, x2_m1, y1_m1, x2_m1, y2_m1, x1_m1, y2_m1],
            [x1_r1, y1_r1, x2_r1, y1_r1, x2_r1, y2_r1, x1_r1, y2_r1],
            [x1_m2, y1_m2, x2_m2, y1_m2, x2_m2, y2_m2, x1_m2, y2_m2],
            [x1_r2, y1_r2, x2_r2, y1_r2, x2_r2, y2_r2, x1_r2, y2_r2],
            [x1_b, y1_b, x2_b, y1_b, x2_b, y2_b, x1_b, y2_b],
            [x1_t, y1_t, x2_t, y1_t, x2_t, y2_t, x1_t, y2_t]
        ]
        random.shuffle(vung_dat_phat_sinh)
        self.toa_do_hien_tai = {}
        for idx, bang in enumerate(BANGS): self.toa_do_hien_tai[bang] = vung_dat_phat_sinh[idx]
        self.tu_dong_tinh_giap_ranh_hinh_hoc()

    def tu_dong_tinh_giap_ranh_hinh_hoc(self):
        self.giap_ranh_dong = {bang: [] for bang in BANGS}
        sai_so_tiep_xuc = 5 
        for i in range(len(BANGS)):
            for j in range(i + 1, len(BANGS)):
                b1 = BANGS[i]; b2 = BANGS[j]
                box1 = self.toa_do_hien_tai[b1]; box2 = self.toa_do_hien_tai[b2]
                min_x1, min_y1 = box1[0], box1[1]; max_x1, max_y1 = box1[4], box1[5]
                min_x2, min_y2 = box2[0], box2[1]; max_x2, max_y2 = box2[4], box2[5]
                giao_x = not (max_x1 < min_x2 - sai_so_tiep_xuc or min_x1 > max_x2 + sai_so_tiep_xuc)
                giao_y = not (max_y1 < min_y2 - sai_so_tiep_xuc or min_y1 > max_y2 + sai_so_tiep_xuc)
                if giao_x and giao_y:
                    self.giap_ranh_dong[b1].append(b2); self.giap_ranh_dong[b2].append(b1)

    def ve_ban_do(self):
        self.canvas.delete("all")
        self.cac_vung_giao_dien = {}
        for bang in BANGS:
            toa_do = self.toa_do_hien_tai[bang]; mau_hien_tai = self.phan_bo_mau[bang]; mau_nen = MAU_HEX[mau_hien_tai]
            poly_id = self.canvas.create_polygon(toa_do, fill=mau_nen, outline="#2c3e50", width=2)
            xs = toa_do[0::2]; ys = toa_do[1::2]
            cx = sum(xs) / len(xs); cy = sum(ys) / len(ys)
            mau_chu = "white" if mau_hien_tai in ["Red", "Blue"] else "black"
            txt_id = self.canvas.create_text(cx, cy, text=bang, font=("Arial", 11, "bold"), fill=mau_chu)
            self.cac_vung_giao_dien[bang] = (poly_id, txt_id)

    def cap_nhat_mau_nhanh(self, bang):
        mau_hien_tai = self.phan_bo_mau[bang]; mau_nen = MAU_HEX[mau_hien_tai]; poly_id, txt_id = self.cac_vung_giao_dien[bang]
        self.canvas.itemconfig(poly_id, fill=mau_nen)
        mau_chu = "white" if mau_hien_tai in ["Red", "Blue"] else "black"
        self.canvas.itemconfig(txt_id, fill=mau_chu); self.cua_so.update()

    def kiem_tra_dung_va_tre(self):
        while self.dang_tam_dung:
            if not self.dang_chay: return False
            self.cua_so.update(); time.sleep(0.1)
        if not self.dang_chay: return False
        return True

    def and_or_search_stochastic(self, mien_gia_tri_hien_tai):
        """THUẬT TOÁN: Forward Checking (Có giới hạn 30 bước)"""
        if self.so_buoc >= 30:
            self.bi_ket = True
            return False

        da_to_xong = True
        for b in BANGS:
            if self.phan_bo_mau[b] is None: da_to_xong = False; break
        if da_to_xong: return True

        if not self.kiem_tra_dung_va_tre(): return False

        bang_hien_tai = None
        for b in BANGS:
            if self.phan_bo_mau[b] is None: bang_hien_tai = b; break

        danh_sach_mau_kha_dung = list(mien_gia_tri_hien_tai[bang_hien_tai])
        random.shuffle(danh_sach_mau_kha_dung)

        for mau in danh_sach_mau_kha_dung:
            if not self.kiem_tra_dung_va_tre(): return False
            
            self.so_buoc += 1
            if self.so_buoc > 30:
                self.bi_ket = True
                return False

            if self.kiem_tra_hop_le(bang_hien_tai, mau):
                self.phan_bo_mau[bang_hien_tai] = mau
                self.cap_nhat_mau_nhanh(bang_hien_tai)
                self._ghi_log_khoi(self.so_buoc, bang_hien_tai, "FORWARD-CK", f"Thử màu {mau} & Quét láng giềng")
                time.sleep(0.4)

                mien_gia_tri_moi = {b: list(mien_gia_tri_hien_tai[b]) for b in BANGS}
                nhanh_hop_le = True

                for hx in self.giap_ranh_dong[bang_hien_tai]:
                    if self.phan_bo_mau[hx] is None:
                        if mau in mien_gia_tri_moi[hx]: mien_gia_tri_moi[hx].remove(mau)
                        if len(mien_gia_tri_moi[hx]) == 0:
                            nhanh_hop_le = False
                            self._ghi_mot_dong_log(f"-> Phát hiện sớm bế tắc: Vùng {hx} sẽ mất hết màu hợp lệ!")
                            break

                if nhanh_hop_le:
                    self._ghi_mot_dong_log(f"-> Nhìn trước OK. Đệ quy tiến sâu...")
                    if self.and_or_search_stochastic(mien_gia_tri_moi): return True

                if self.bi_ket or not self.kiem_tra_dung_va_tre(): return False

                self._ghi_mot_dong_log(f"<- Thất bại nhánh nhìn trước! Lùi lại tại vùng: {bang_hien_tai}")
                self.phan_bo_mau[bang_hien_tai] = None
                self.cap_nhat_mau_nhanh(bang_hien_tai)
                self._ghi_log_khoi(self.so_buoc, bang_hien_tai, "BACKTRACK", "Hủy màu, quay lui!")
                time.sleep(0.2)
            else:
                self._ghi_mot_dong_log(f"Xung đột ranh giới thực tế: {bang_hien_tai} trùng màu {mau}")

        return False

    def kiem_tra_hop_le(self, bang, mau):
        for hx in self.giap_ranh_dong[bang]:
            if self.phan_bo_mau[hx] == mau: return False
        return True

    def dao_trang_thai_dung(self):
        if not self.dang_chay: return
        if self.dang_tam_dung:
            self.dang_tam_dung = False; self.btn_dung.config(text="Dừng thuật toán", bg="#e74c3c")
            self.nhan_trang_thai.config(text="Thuật toán đang tiếp tục chạy...", fg="blue")
        else:
            self.dang_tam_dung = True; self.btn_dung.config(text="Tiếp tục chạy", bg="#27ae60")
            self.nhan_trang_thai.config(text="Đã DỪNG thuật toán khẩn cấp!", fg="#d35400")

    def bat_dau_thuat_toan(self):
        if self.dang_chay: return
        self.dang_chay = True; self.dang_tam_dung = False; self.so_buoc = 0; self.bi_ket = False
        self.btn_chay.config(state="disabled"); self.btn_dung.config(state="normal", text="Dừng thuật toán", bg="#e74c3c")
        self.o_chu_log.config(state="normal"); self.o_chu_log.delete(1.0, tk.END); self.o_chu_log.config(state="disabled")
        self.nhan_trang_thai.config(text="Forward Checking đang quét lọc và thu hẹp miền giá trị...", fg="blue")
        
        for b in BANGS: self.phan_bo_mau[b] = None; self.cap_nhat_mau_nhanh(b)
            
        mien_ban_dau = {b: list(MAU_SAC) for b in BANGS}
        success = self.and_or_search_stochastic(mien_ban_dau)
        
        if not self.dang_chay:
            self.nhan_trang_thai.config(text="Đã dừng tiến trình cũ.", fg="black")
        elif self.bi_ket:
            self.nhan_trang_thai.config(text=f"Thông báo: Đã vượt quá giới hạn {self.so_buoc} bước! Bị kẹt không thể hoàn thành.", fg="#e67e22", font=("Arial", 11, "bold"))
            self._ghi_mot_dong_log("⚠️ THÔNG BÁO: Quá 30 bước toán! Thuật toán bị kẹt khẩn cấp.")
        elif success:
            self.nhan_trang_thai.config(text=f"Thành công! Tìm ra lời giải sau {self.so_buoc} bước toán.", fg="green")
        else:
            self.nhan_trang_thai.config(text="Thất bại: Cấu hình địa hình ngẫu nhiên này vô nghiệm.", fg="red")
            
        self.dang_chay = False; self.dang_tam_dung = False
        self.btn_chay.config(state="normal"); self.btn_dung.config(state="disabled", text="Dừng thuật toán", bg="#e74c3c")

    def dat_lai_va_sinh_dia_hinh(self):
        self.dang_chay = False; self.dang_tam_dung = False; self.phan_bo_mau = {bang: None for bang in BANGS}; self.so_buoc = 0; self.bi_ket = False
        self.sinh_ban_do_so_le_tu_nhien(); self.ve_ban_do()
        self.o_chu_log.config(state="normal"); self.o_chu_log.delete(1.0, tk.END); self.o_chu_log.config(state="disabled")
        self.btn_chay.config(state="normal"); self.btn_dung.config(state="disabled", text="Dừng thuật toán", bg="#e74c3c")
        self.nhan_trang_thai.config(text="Đã ĐẶT LẠI: Đổi vị trí + Sinh địa hình so le + Quét lại ranh giới hình học!", fg="black")

    def _ghi_log_khoi(self, buoc, vung, hanh_dong, ket_qua):
        self.o_chu_log.config(state="normal")
        m = self.phan_bo_mau
        trang_thai_str = (
            f"│ WA:{m['WA'] or '?':<5} NT:{m['NT'] or '?':<5} SA:{m['SA'] or '?':<5} │\n"
            f"│ Q :{m['Q'] or '?':<5} NSW:{m['NSW'] or '?':<5} V :{m['V'] or '?':<5} │\n"
            f"│ T :{m['T'] or '?':<5} {'':19} │"
        )
        log_txt = (
            f"┌─ Bước {buoc:02d} ──────────────────────────┐\n"
            f"│ Vùng: {vung:<10} | Kiểu: {hanh_dong:<10} │\n"
            f"│ Kết quả: {ket_qua:<25} │\n"
            f"{trang_thai_str}\n"
            f"└──────────────────────────────────────┘\n"
            f"                    ▼                    \n"
        )
        self.o_chu_log.insert(tk.END, log_txt); self.o_chu_log.see(tk.END); self.o_chu_log.config(state="disabled")

    def _ghi_mot_dong_log(self, txt):
        self.o_chu_log.config(state="normal"); self.o_chu_log.insert(tk.END, f"  > {txt}\n"); self.o_chu_log.config(state="disabled")

if __name__ == "__main__":
    root = tk.Tk()
    app = GiaoDienToMauAndOrGiatCap(root)
    root.mainloop()